# **Recommendation systems, ICA, and matrix factorization**

**Libraries**
-

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.models.latent_factors import FactorData, FASearch, ICASearch, FactorAnalysisResult, FactorComparison
from src.models.recommendation import RecommendationData, RecommendationSearch, SVDRecommender
from src.models.latent_factor_plots import (
    plot_fa_search, plot_ica_search, plot_fa_loadings, plot_ica_loadings,
    plot_factor_comparison, plot_recommendation_search, plot_product_popularity,
)


## **1. Question of Interest**

### **1.2 Question**

- To what extent can Factor Analysis (FA), PCA, and ICA identify interpretable latent factors underlying commercial sales patterns, how do their representations differ, and how effectively can matrix factorization support customer–product recommendation systems?

### **1.2 Tools that are going to be used**

To continue identifiying latent factors underlying commercial patterns, we are gonna use the next models:
| Criterion           | Factor Analysis (FA)                          | PCA                                       | ICA                                          |
| ------------------- | --------------------------------------------- | ----------------------------------------- | -------------------------------------------- |
| Main objective      | Identify underlying latent factors.           | Maximize explained variance.              | Identify independent latent sources.         |
| Approach            | Probabilistic factor model.                   | Variance-based decomposition.             | Statistical source separation.               |
| Key assumption      | Shared factors explain observed correlations. | High-variance directions are informative. | Sources are statistically independent.       |
| Factor relationship | Correlated or uncorrelated.                   | Orthogonal and uncorrelated.              | Statistically independent.                   |
| Interpretability    | Often easier to interpret.                    | Not necessarily interpretable.            | May reveal distinct hidden patterns.         |
| Main limitation     | Depends on modeling assumptions.              | Variance does not imply relevance.        | Independence assumptions may be unrealistic. |

On the other hand, for implementing a customer Truncated SVD is going to be applied from a customer-product interaction matrix, and finally the recommended system implemented will be compared with the popularity baseline to evaluate performance.

## **2. Database Loading and train/val/test**

### **2.1 Reading CSV**

In [ ]:
factor_data = FactorData()
factor_data.frame.head()


In [ ]:
factor_data.X.head()


### **2.2 Split**

In [ ]:
display(factor_data.split_summary())


### **3.3 Standarize**

In [ ]:
factor_data.scaler


## **3. Factor Analysis (FA)**

### **3.1 Notation**

FA stands for **Factor Analysis**. It is a probabilistic latent variable model that assumes the observed features can be explained by a smaller number of unobserved factors and Gaussian noise.

FA uses a similar structure to PPCA:

$$
\boxed{X_c=ZW^\top+E}
$$

Where:

- $X_c\in\mathbb R^{n\times p}$ is the centered data matrix.
- $Z\in\mathbb R^{n\times k}$ contains the unobserved latent factors.
- $W\in\mathbb R^{p\times k}$ is the factor loading matrix.
- $E\in\mathbb R^{n\times p}$ represents Gaussian noise.

As in PPCA, we assume that the latent factors follow a standard multivariate Gaussian distribution:

$$
Z_{i,:}^\top\sim\mathcal N(0,I_k)
$$

However, **the main difference between FA and PPCA is the assumption about the noise covariance**.

In PPCA, every feature has the same noise variance, whereas FA allows each feature to have its own noise variance:

$$
\boxed{E_{i,:}^\top\sim\mathcal N(0,\Psi)}
$$

Where:

$$
\Psi=
\begin{bmatrix}
\psi_1&0&\cdots&0\\
0&\psi_2&\cdots&0\\
\vdots&\vdots&\ddots&\vdots\\
0&0&\cdots&\psi_p
\end{bmatrix}
$$

Here, $\psi_j$ represents the specific noise variance associated with feature $j$. The diagonal structure assumes that the noise terms of different features are uncorrelated.

Since the latent factors and noise are independent Gaussian random variables, their linear combination is also Gaussian. Therefore:

$$
\boxed{X_{c,i,:}^\top\sim\mathcal N(0,C)}
$$

With covariance matrix:

$$
\boxed{C=WW^\top+\Psi}
$$

Where:

- $WW^\top$ represents the covariance explained by the common latent factors.
- $\Psi$ represents the feature-specific residual variances, also called *uniquenesses*.
- $C$ represents the covariance matrix assumed by the probabilistic model.

The main difference between PPCA and FA can therefore be summarized as:

$$
\underbrace{C_{\text{PPCA}}=WW^\top+\sigma^2I_p}_{\text{Equal noise variance}}
$$

$$
\underbrace{C_{\text{FA}}=WW^\top+\Psi}_{\text{Feature-specific noise variance}}
$$

Consequently, **FA is more flexible than PPCA in modeling the covariance structure**, because it allows each observed variable to have a different amount of unexplained variance.

Finally, after estimating the model, we can obtain the latent factor scores using their conditional expectation:

$$
\boxed{
\hat Z=X_cC^{-1}W
}
$$

Where $\hat Z$ represents the estimated latent factor scores and $C^{-1}$ is the inverse of the covariance matrix assumed by the model.

We can then reconstruct an approximation of the original matrix:

$$
\boxed{
\hat X=\hat ZW^\top+\mathbf{1}\mu
}
$$

Where $\mu$ is the vector of feature means and $\mathbf{1}$ is a column vector of ones.

Unlike standard PCA, these factor scores are estimated using the probabilistic model, taking both the common factor structure and the feature-specific noise variances into account.

### **3.2 Evaluating different $k$**

In [ ]:
fa_search = FASearch(n_jobs=-1, force_retrain=False).run(factor_data)
display(fa_search.results.round(6))


### **3.3 Metrics Visualization**

In [ ]:
plot_fa_search(fa_search)


**Observations**

We selected **$k=5$ latent factors** based on the following considerations:

- **Reconstruction quality:** With $k=5$, the model achieves approximately 88.54% reconstructed energy on validation data, with a validation MSE of 0.1170 and a relative Frobenius error of 0.3385.
- **Diminishing improvements:** Reconstruction improves substantially between $k=1$ and $k=5$. Although $k=6$ provides a further improvement, additional factors beyond $k=6$ produce negligible reductions in reconstruction error.
- **Factor stability:** At $k=5$, the mean bootstrap factor similarity is approximately 0.99998, indicating that the estimated factor-loading directions remain highly consistent across resampled datasets.
- **Interpretability and complexity:** Five factors provide a compact representation of the dominant patterns in the data while avoiding unnecessary model complexity.

### **3.4 FA with $k=5$**

In [ ]:
fa_analysis = FactorAnalysisResult(fa_search, k=5).fit()
display(fa_analysis.loadings.round(4))
display(fa_analysis.metrics.round(6))
display(fa_analysis.scores["train"].head())


In [ ]:
plot_fa_loadings(fa_analysis)


**Observations**

- **Factor 1 — Sales Value and Product Price:** This factor is characterized by strong positive loadings for Sales per Customer (0.996), Sales (0.983), and Product Price (0.824), along with a moderate positive loading for Order Item Discount (0.478). It may represent the monetary value of transactions, particularly those involving higher-priced products and larger absolute discounts.

- **Factor 2 — Discounts:** This factor is dominated by strong positive loadings for Order Item Discount (0.878) and Order Item Discount Rate (0.828). It may capture the discount intensity of transactions, identifying observations associated with larger discounts in both absolute and relative terms.

- **Factor 3 — Product Price versus Quantity:** This factor exhibits a strong negative loading for Order Item Quantity (-0.958) and a moderate positive loading for Product Price (0.525). It may represent a contrast between product price and purchased quantity, where higher factor scores are associated with higher-priced products purchased in smaller quantities.

- **Factor 4 — Profitability:** This factor is characterized by strong negative loadings for Order Item Profit Ratio (-0.910) and Benefit per Order (-0.907). Higher factor scores are associated with lower profitability indicators, potentially identifying less profitable transactions. 

- **Factor 5 — Shipping Duration:** This factor is dominated by negative loadings for Days for Shipping (Real) (-0.717) and Days for Shipment (Scheduled) (-0.717). Higher factor scores are associated with shorter actual and scheduled shipping durations.

## **4. Independent Component Analysis (ICA)**

### **4.1 Notation**


ICA stands for **Independent Component Analysis**. It is a statistical technique that assumes the observed variables are generated by linear combinations of latent sources that are statistically independent.

ICA assumes that the centered data matrix can be represented as:

$$
\boxed{X_c=SA^\top}
$$

Where:

- $X_c\in\mathbb R^{n\times p}$ is the centered data matrix.
- $S\in\mathbb R^{n\times k}$ contains the unobserved independent components.
- $A\in\mathbb R^{p\times k}$ is the mixing matrix.

The main objective of ICA is to estimate the independent sources $S$ from the observed data $X_c$.

Unlike PCA, which finds uncorrelated components that maximize variance, **ICA searches for components that are statistically independent**.

This means that the joint probability distribution of the sources can be expressed as:

$$
\boxed{
p(s_1,s_2,\ldots,s_k)=\prod_{j=1}^{k}p(s_j)
}
$$

To identify these independent sources, ICA assumes that they are **non-Gaussian**, with at most one Gaussian component. This is important because mixtures of independent variables tend to be more Gaussian than the original sources under suitable conditions, as suggested by the Central Limit Theorem.

Therefore, ICA searches for directions that maximize non-Gaussianity as a way to recover independent components.

Before estimating these components, ICA typically applies **whitening**, which transforms the centered observations so that their covariance matrix becomes the identity matrix:

$$
\boxed{X_w=X_cV_k\Lambda_k^{-1/2}}
$$

Where $V_k$ contains the selected eigenvectors of the covariance matrix and $\Lambda_k$ contains their corresponding eigenvalues.

After whitening, ICA estimates an unmixing matrix $B$ that produces the independent components:

$$
\boxed{\hat S=X_wB^\top}
$$

The unmixing matrix is estimated through an optimization procedure that maximizes a measure of non-Gaussianity, such as an approximation of negentropy.

Finally, the original data can be approximately reconstructed using the estimated independent components and the mixing matrix:

$$
\boxed{\hat X=\hat SA^\top+\mathbf 1\mu}
$$

Where $\mu$ represents the vector of feature means.

Consequently, **ICA differs from PCA and FA because its primary objective is not to maximize explained variance or model shared covariance, but to identify statistically independent latent sources underlying the observed variables**.


### **4.2 Evaluating different $k$**

In [ ]:
ica_search = ICASearch(n_jobs=-1, force_retrain=False).run(factor_data)
display(ica_search.results.round(6))
display(ica_search.results[["k", "Converged", "Iterations"]])


### **4.3 Metrics Visualization**

In [ ]:
plot_ica_search(ica_search)


**Observations**

We selected **$k=5$ independent components** based on the following considerations:

- **Reconstruction quality:** With $k=5$, ICA achieves approximately 92.26% reconstructed energy on validation data, indicating that five components retain a substantial proportion of the original data's squared norm.
- **Reconstruction error:** The validation MSE decreases to approximately 0.0790, while the relative Frobenius error reaches approximately 0.2782. Although additional components improve reconstruction, five components provide a relatively compact representation.
- **Component stability:** The mean bootstrap component similarity is approximately 0.9880, indicating that the estimated mixing directions are highly consistent across resampled training datasets.
- **Component concentration:** The mean concentration is approximately 0.4805, suggesting that the components exhibit moderately concentrated mixing coefficients across the original variables.
- **Model complexity:** Although larger configurations improve reconstruction, five components were selected to balance dimensionality reduction, reconstruction quality, and stability.


### **4.4 Independent Component Analysis (ICA) $k=5$**

In [ ]:
ica_analysis = FactorAnalysisResult(ica_search, k=5).fit()
display(ica_analysis.loadings.round(4))
display(ica_analysis.metrics.round(6))
display(ica_analysis.kurtosis.round(6))
display(ica_analysis.scores["train"].head())
plot_ica_loadings(ica_analysis)


**Observations**

- **IC 1 — Discount Intensity:** This component is dominated by strong positive mixing coefficients for Order Item Discount Rate (0.977) and Order Item Discount (0.702). Higher component scores are associated with higher discount rates and larger absolute discounts, suggesting that this component captures discount-related purchasing patterns.

- **IC 2 — Profitability:** This component is characterized by strong negative mixing coefficients for Order Item Profit Ratio (-0.950) and Benefit per Order (-0.923). Higher component scores are associated with lower profitability indicators, potentially representing transactions with reduced profitability. However, this does not necessarily imply that these transactions generate losses.

- **IC 3 — Price–Quantity Contrast:** This component exhibits a strong negative coefficient for Order Item Quantity (-0.996) and a moderate positive coefficient for Product Price (0.494). Higher component scores are associated with lower purchased quantities and higher product prices, suggesting a contrast between product price and purchase volume.

- **IC 4 — Shipping Duration:** This component is dominated by strong positive coefficients for Days for Shipping (Real) (0.870) and Days for Shipment (Scheduled) (0.870). Higher component scores are associated with longer actual and scheduled shipping durations, potentially capturing variation in overall delivery times.

- **IC 5 — Transaction Value and Discounts:** This component is characterized by strong positive coefficients for Sales (0.982), Sales per Customer (0.963), and Product Price (0.849), along with a substantial positive coefficient for Order Item Discount (0.654). Higher component scores are associated with greater transaction values, higher-priced products, and larger absolute discounts.

## **5. Principal Component Analysis (PCA)** 

Remembering the things were done in 03_svd_pca_probabilistic_lsa.ipynb, we could identify some latent variables in PCA with $k=5$ components that resulted in:

- **Component 1 — High Sales and Discounts:** This component appears to capture positive contributions from Sales per customer, order item discount, sales and product price. It appears to involve products that have an elevated price, greater sales values, and larger absolute discount amounts.

- **Component 2 — Profitability:** This component is dominated by Benefit per order and Order Item Profit Ratio, where we can suggest that this component represents the orders that have high profitability.

- **Component 3 — Shipping Duration:** This component is dominated Days for shipping (real) and Days for shipment (schedule), it appears to describe the orders with longer scheduled and actual shipping durations.

- **Component 4 — Discounts and Relatively Low Sales:** This component captures positive and higher contributions of Order Item Discount Rate and Order Item Discount while small and negative contributions with Sales, Product Price and Sales per costumer. This component could represent the orders that have a high discount, but they get relatively lower sales values to the company.

- **Component 5 - Quantity:** This component has a strong positive contribution with Order Item quantity, a positive but not strong with sales and sales per costumer, but a negative contribution with order item discount rate and product price. This suggest that the component could represent the orders that usually have a low price per product without a high discount, but they have a larger quantities per order, associated with modestly higher sales values.

## **6. ICA vs FA vs PCA Final Comparison**

### **6.1 Metrics and Latent Variables**

In [ ]:
factor_comparison = FactorComparison(fa_analysis, ica_analysis, pca_k=5).run()
plot_factor_comparison(factor_comparison)


**Observations — Comparison of PCA, FA, and ICA**

- **Similar latent patterns:** The three methods identify comparable commercial dimensions associated with transaction value, profitability, discount intensity, shipping duration, and purchased quantity.

- **FA and ICA similarity:** Factor Analysis and ICA produce particularly similar loading and mixing patterns, with several components dominated by nearly the same original variables. This suggests that both methods recover comparable underlying structures despite their different statistical objectives.

- **PCA differences:** PCA identifies similar commercial patterns but distributes some variables differently across its components. In particular, the relationship between product price and purchased quantity is represented differently than in FA and ICA, reflecting PCA's objective of maximizing explained variance.

- **Interpretability:** FA and ICA provide relatively distinct components associated with specific commercial characteristics. PCA also produces interpretable components, although some represent broader combinations of variables.

- **Overall conclusion:** The results suggest that the dominant commercial patterns are consistently identifiable across the three methods. However, FA and ICA provide more closely aligned representations, while PCA emphasizes directions of maximum variance rather than the separation of latent sources.

### **6.2 Quality**

In [ ]:
display(factor_comparison.results.round(6))


**Observations**

- PCA and ICA achieve lower reconstruction errors than Factor Analysis (FA), indicating that they preserve more information from the original dataset when reducing its dimensionality to five components. However, this does not necessarily imply that they produce more useful or interpretable latent representations.

- The latent factor analysis reveals that ICA and FA identify highly similar components associated with commercial characteristics, including transaction value, profitability, discount intensity, shipping duration, and the price–quantity relationship. PCA identifies comparable patterns, although some of its components combine contributions from a broader set of original variables.

- The relatively low reconstruction errors indicate that the models provide good approximations of the original data while preserving relevant information. Additionally, the extracted latent factors reveal meaningful transactional patterns, supporting the quality and interpretability of the resulting representations.

### **6.3 Utility and Interpretability**

The extracted latent factors have potential utility for commercial analysis, as they summarize transaction value, profitability, discount intensity, shipping duration, and the relationship between product prices and purchased quantities. These dimensions could support transaction segmentation, promotional analysis, pricing strategies, and logistics-related decision-making.

Potential applications include:

- **Transaction Value:** Segment transactions into different monetary-value groups to identify purchasing patterns and distinguish between low-value and high-value transactions.
- **Profitability:** Identify transactions associated with lower or higher profitability, supporting the development of strategies to improve financial performance.
- **Discount Intensity:** Analyze promotional patterns and examine how purchasing behavior varies across transactions with different discount levels.
- **Shipping Duration:** Identify orders with longer or shorter shipping times to support the development of more efficient delivery strategies.
- **Price–Quantity Relationship:** Analyze purchasing patterns involving product prices and quantities sold, providing insights that could support pricing and sales strategies.

On the other hand, ICA and FA provide relatively interpretable latent representations, with several components strongly associated with specific groups of commercial variables. PCA identifies similar patterns, although some principal components combine contributions from a broader set of features.

Finally, the consistency of the main commercial dimensions across the three algorithms supports the interpretability of the extracted representations, suggesting that these patterns are not exclusive to a single decomposition method.



## **7. Recommender System**

In [ ]:
recommendation_data = RecommendationData(factor_data.frame)


### **7.1 Customer-Product Interaction Matrix**

In [ ]:
print("Interaction matrix:", recommendation_data.matrix.shape)
print("Observed interactions:", recommendation_data.matrix.nnz)


In [ ]:
display(recommendation_data.activity_summary())


### **7.2 Train/Val/Test Split**

In [ ]:
recommendation_data.split()
display(recommendation_data.split_summary.round(2))


### **7.3 Truncated SVD with different $k$**

We approximate the customer–product interaction matrix as:
$$
R \approx U_k\Sigma_k V_k^\top
$$

Where:
- $U_k$: Customer latent factors.
- $\Sigma_k$: Singular values.
- $V_k$: Product latent factors.
- $k$: Number of latent dimensions.
In scikit-learn, fit_transform() returns $U_k\Sigma_k$, while components_.T corresponds to $V_k$.R

In [ ]:
recommendation_search = RecommendationSearch().run(recommendation_data)
display(recommendation_search.results.round(6))


#### **7.3.1 Metrics Visualization**

In [ ]:
plot_recommendation_search(recommendation_search)


**Observations**

- $k=2$ is the best option among the evaluated values, as it maximizes both HitRate@10 and NDCG@10. The HitRate@10 is approximately 82.53%, indicating that the system successfully includes the held-out product among its top 10 recommendations for nearly 83% of evaluated customers. On the other hand, NDCG@10 reaches approximately 0.54, indicating moderate ranking quality. Although the system frequently identifies the relevant product, its position within the recommendation list can still be improved. Nevertheless, $k=2$ achieves the best ranking performance among the evaluated configurations.

### **7.4 Latent Factors**

In [ ]:
recommender = SVDRecommender(k=2, top_k=10).fit(recommendation_data)
product_factors = recommender.product_factors()
display(recommender.factor_extremes())
display(product_factors.select_dtypes("number").corr())


**Observations**

- Factor 1 presents a strong positive correlation with product popularity ((r=0.8526)), suggesting that this latent dimension primarily captures the dominant interaction patterns associated with frequently purchased products.
- Factor 2 has almost a nule correlation with popularity ((r=-0.0716)) and presents contrasting coefficients among highly popular products. Perfect Fitness Rip Deck has a strong positive coefficient ((0.7629)), while Nike Golf Polo ((-0.5496)) and O'Brien Life Vest ((-0.3265)) have strong negative coefficients. Factor 2 could be identifying differences in interaction patterns on some popular products.
- Since both factors are strongly influenced by a small group of highly interacted-with products, the imbalance in the interaction matrix may be limiting the model's ability to identify more diverse preference patterns.

### **7.5 Evaluation of the System with $k=2$**

#### **7.5.1 Recommender System Implemented vs Popularity Baseline**

In [ ]:
display(recommender.evaluate().round(6))


**Observations**

- The Popularity Baseline achieved higher performance metrics than the implemented recommendation system based on Truncated SVD. One possible explanation is the imbalance in the customer–product interaction matrix, where certain products are purchased much more frequently than others. Consequently, recommending the most popular products can be more effective than using the latent factors identified by SVD to predict customer preferences.

#### **7.5.2 Imbalance of Products**

In [ ]:
display(recommender.popularity_summary().round(2))
display(recommender.popularity.describe())
plot_product_popularity(recommender)


**Observations**

- The product popularity distribution confirms a substantial imbalance in the customer–product interaction matrix, with the top 20 products accounting for approximately 82% of all observed interactions. This indicates that customer purchases are highly concentrated in a relatively small group of products. Consequently, recommending the most popular products represents an effective strategy for this dataset, which may explain why the Popularity Baseline outperformed the Truncated SVD recommendation system.